# TEJ Wizard vs FinMind：還原股價交集比對

> **這份 notebook 的結果是 2026-09-03 的快照，尚未在擴充後的 universe 上重跑。**
> 當時 TEJ 側只有 120 檔 workbook（0050 歷史成分聯集 119 檔 + ETF）；
> 2026-09-05 已擴充成 367 檔（TWN50 ∪ TM100 成分 365 檔 + 0050、0051），
> 列數從 651,118 增為 1,887,747。因此下面所有「共同股票數／列數」都偏低，
> 而**調整方法的比較結論不受影響**——那是逐檔逐日比對得到的，
> 換更大的樣本只會增加證據，不會反轉方向。要更新數字就整份重跑。

本 notebook 比較兩個 DuckDB 的 `v_adj_daily_prices`，只在 **相同 `stock_id`、相同 `date`** 的紀錄上比對。資料庫皆以 `READ_ONLY` attach，不會修改原始 DB。

比對規則：

- adjusted OHLC：同時報告完全相等，以及絕對差不超過 `0.01` 元的結果。完全相等很容易受到 TEJ 4 位小數與 FinMind 6 位小數的精度差影響。
- volume：TEJ `volume_k_shares` 是千股，先乘 1,000 後與 FinMind `volume`（股）比較；TEJ 的千股欄位有取整，因此另報告差不超過 500 股。
- turnover：TEJ `turnover_k` 是千元，先乘 1,000 後與 FinMind `turnover_value`（元）比較；另報告差不超過 500 元。
- transactions：直接比較筆數。
- `spread` 與 `price_change` 沒有在文件中證實為完全相同定義，因此不混入主比對。

> 注意：兩個來源都稱為 back-adjusted，但公司行動的調整方法、調整基準日與更新時間可能不同；歷史 OHLC 不同不一定代表其中一筆是輸入錯誤。

若 kernel 尚未安裝繪圖套件，可在 `data-pipelines/tej-wizard` 執行 `pip install -e .[notebook]`。

In [ ]:
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.figsize": (10, 5), "axes.spines.top": False, "axes.spines.right": False})

ABS_PRICE_TOL = 0.01       # NTD
VOLUME_ROUND_TOL = 500     # shares
TURNOVER_ROUND_TOL = 500   # NTD

def find_repo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "db/tej-wizard/tej_wizard.duckdb").exists() and (candidate / "db/finmind/finmind.duckdb").exists():
            return candidate
    raise FileNotFoundError("找不到 financial-db repo root；請從 repo 內啟動 notebook。")

ROOT = find_repo_root(Path.cwd())
TEJ_DB = ROOT / "db/tej-wizard/tej_wizard.duckdb"
FINMIND_DB = ROOT / "db/finmind/finmind.duckdb"
ROOT, TEJ_DB, FINMIND_DB

## 1. 唯讀連線與資料概況

In [ ]:
con = duckdb.connect()  # in-memory；TEMP TABLE 只存在這個 session
con.execute("PRAGMA threads=4")
con.execute(f"ATTACH '{TEJ_DB.as_posix()}' AS tej (READ_ONLY)")
con.execute(f"ATTACH '{FINMIND_DB.as_posix()}' AS finmind (READ_ONLY)")

coverage = con.execute("""
SELECT 'TEJ Wizard' AS dataset, count(*) AS row_count,
       count(DISTINCT stock_id) AS stock_count,
       min(date) AS min_date, max(date) AS max_date
FROM tej.v_adj_daily_prices
UNION ALL
SELECT 'FinMind', count(*), count(DISTINCT stock_id), min(date), max(date)
FROM finmind.v_adj_daily_prices
""").df()
display(coverage)

In [ ]:
# 先確認 join key 在兩邊都沒有重複，避免 many-to-many join 放大筆數。
key_quality = con.execute("""
SELECT 'TEJ Wizard' AS dataset,
       count(*) - count(DISTINCT (stock_id, date)) AS duplicate_keys
FROM tej.v_adj_daily_prices
UNION ALL
SELECT 'FinMind', count(*) - count(DISTINCT (stock_id, date))
FROM finmind.v_adj_daily_prices
""").df()
display(key_quality)
assert key_quality["duplicate_keys"].eq(0).all(), "發現重複 key，停止比對以免 join 放大。"

## 2. 建立相同股票、相同日期的交集

交集只保留這次會比較的欄位，並將 TEJ 的 volume/turnover 單位轉成與 FinMind 相同。

In [ ]:
con.execute("""
CREATE OR REPLACE TEMP TABLE overlap AS
SELECT
    t.stock_id, t.stock_name, t.date,
    t.open AS tej_open, f.open AS finmind_open,
    t.high AS tej_high, f.high AS finmind_high,
    t.low AS tej_low, f.low AS finmind_low,
    t.close AS tej_close, f.close AS finmind_close,
    t.volume_k_shares * 1000 AS tej_volume, f.volume AS finmind_volume,
    t.turnover_k * 1000 AS tej_turnover, f.turnover_value AS finmind_turnover,
    t.transactions AS tej_transactions, f.transactions AS finmind_transactions,
    t.ingested_at AS tej_ingested_at, f.ingested_at AS finmind_ingested_at,
    f.source AS finmind_source
FROM tej.v_adj_daily_prices AS t
INNER JOIN finmind.v_adj_daily_prices AS f USING (stock_id, date)
""")

overlap_summary = con.execute("""
WITH common_stocks AS (
    SELECT stock_id FROM tej.v_adj_daily_prices
    INTERSECT
    SELECT stock_id FROM finmind.v_adj_daily_prices
),
tej_common AS (
    SELECT count(*) AS n FROM tej.v_adj_daily_prices
    SEMI JOIN common_stocks USING (stock_id)
),
finmind_common AS (
    SELECT count(*) AS n FROM finmind.v_adj_daily_prices
    SEMI JOIN common_stocks USING (stock_id)
)
SELECT
    count(*) AS overlap_rows,
    count(DISTINCT stock_id) AS overlap_stocks,
    min(date) AS first_overlap_date,
    max(date) AS last_overlap_date,
    (SELECT n FROM tej_common) - count(*) AS tej_only_dates_on_common_stocks,
    (SELECT n FROM finmind_common) - count(*) AS finmind_only_dates_on_common_stocks
FROM overlap
""").df()
display(overlap_summary)

In [ ]:
# TEJ 有、但 FinMind adjusted price 完全沒有的股票。
tej_only_stocks = con.execute("""
SELECT s.stock_id, s.stock_name, s.first_date, s.last_date, s.trading_days
FROM tej.v_securities AS s
WHERE NOT EXISTS (
    SELECT 1 FROM finmind.v_adj_daily_prices AS f WHERE f.stock_id = s.stock_id
)
ORDER BY s.stock_id
""").df()
display(tej_only_stocks)

## 3. Adjusted OHLC 差異

先看各欄位，再看一整列四個價位是否都相同。相對差異以 FinMind 的絕對值為分母；分母為 0 時略過。

In [ ]:
ohlc_metrics = con.execute(f"""
WITH long AS (
    SELECT 'open' AS field, tej_open AS tej_value, finmind_open AS finmind_value FROM overlap
    UNION ALL SELECT 'high', tej_high, finmind_high FROM overlap
    UNION ALL SELECT 'low', tej_low, finmind_low FROM overlap
    UNION ALL SELECT 'close', tej_close, finmind_close FROM overlap
)
SELECT
    field, count(*) AS n,
    count(*) FILTER (WHERE tej_value IS NULL) AS tej_nulls,
    count(*) FILTER (WHERE finmind_value IS NULL) AS finmind_nulls,
    count(*) FILTER (WHERE tej_value = finmind_value) AS exact_equal,
    round(100.0 * exact_equal / n, 4) AS exact_equal_pct,
    count(*) FILTER (WHERE abs(tej_value - finmind_value) <= {ABS_PRICE_TOL}) AS within_tolerance,
    round(100.0 * within_tolerance / n, 4) AS within_tolerance_pct,
    round(median(abs(tej_value - finmind_value)), 6) AS median_abs_diff,
    round(quantile_cont(abs(tej_value - finmind_value), 0.95), 6) AS p95_abs_diff,
    round(max(abs(tej_value - finmind_value)), 6) AS max_abs_diff,
    round(median(abs(tej_value - finmind_value) / nullif(abs(finmind_value), 0)) * 100, 6) AS median_abs_pct_diff,
    round(quantile_cont(abs(tej_value - finmind_value) / nullif(abs(finmind_value), 0), 0.95) * 100, 6) AS p95_abs_pct_diff,
    round(max(abs(tej_value - finmind_value) / nullif(abs(finmind_value), 0)) * 100, 6) AS max_abs_pct_diff
FROM long
GROUP BY field
ORDER BY field
""").df()
display(ohlc_metrics)

In [ ]:
row_metrics = con.execute(f"""
SELECT
    count(*) AS n,
    count(*) FILTER (WHERE
        tej_open = finmind_open AND tej_high = finmind_high
        AND tej_low = finmind_low AND tej_close = finmind_close
    ) AS all_ohlc_exact,
    round(100.0 * all_ohlc_exact / n, 4) AS all_ohlc_exact_pct,
    count(*) FILTER (WHERE
        abs(tej_open - finmind_open) <= {ABS_PRICE_TOL}
        AND abs(tej_high - finmind_high) <= {ABS_PRICE_TOL}
        AND abs(tej_low - finmind_low) <= {ABS_PRICE_TOL}
        AND abs(tej_close - finmind_close) <= {ABS_PRICE_TOL}
    ) AS all_ohlc_within_tolerance,
    round(100.0 * all_ohlc_within_tolerance / n, 4) AS all_ohlc_within_tolerance_pct,
    n - all_ohlc_within_tolerance AS rows_different_over_tolerance
FROM overlap
""").df()
display(row_metrics)

In [ ]:
# Close 相對差異分布：用互斥區間避免少數極端值壓扁整張圖。
close_diff_values = con.execute("""
SELECT abs(tej_close - finmind_close) / nullif(abs(finmind_close), 0) * 100 AS abs_pct_diff
FROM overlap
WHERE finmind_close <> 0
""").df()["abs_pct_diff"]

bins = [-np.inf, 0.001, 0.01, 0.1, 1, 10, np.inf]
labels = ["≤0.001%", "0.001–0.01%", "0.01–0.1%", "0.1–1%", "1–10%", ">10%"]
dist = pd.cut(close_diff_values, bins=bins, labels=labels, include_lowest=True).value_counts(sort=False)
dist_pct = dist / dist.sum() * 100

fig, ax = plt.subplots(figsize=(10, 4.8))
bars = ax.bar(dist_pct.index.astype(str), dist_pct.values, color="#4C78A8")
ax.bar_label(bars, fmt="%.1f%%", padding=3)
ax.set(title="Close absolute percentage-difference distribution", xlabel="|TEJ − FinMind| / |FinMind|", ylabel="Share of overlapping rows (%)")
ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

## 4. 差異隨時間與股票的分布

In [ ]:
by_year = con.execute(f"""
SELECT
    year(date) AS calendar_year, count(*) AS n,
    count(*) FILTER (WHERE abs(tej_close - finmind_close) > {ABS_PRICE_TOL}) AS close_diff_rows,
    round(100.0 * close_diff_rows / n, 3) AS close_diff_pct,
    round(median(abs(tej_close - finmind_close)), 6) AS median_abs_diff,
    round(median(abs(tej_close - finmind_close) / nullif(abs(finmind_close), 0)) * 100, 6) AS median_abs_pct_diff,
    round(quantile_cont(abs(tej_close - finmind_close) / nullif(abs(finmind_close), 0), 0.95) * 100, 6) AS p95_abs_pct_diff
FROM overlap
GROUP BY calendar_year
ORDER BY calendar_year
""").df()
display(by_year)

In [ ]:
fig, ax1 = plt.subplots(figsize=(11, 5))
ax2 = ax1.twinx()
line1 = ax1.plot(by_year["calendar_year"], by_year["close_diff_pct"], marker="o", color="#E45756", label="Rows with close diff > NT$0.01")
line2 = ax2.plot(by_year["calendar_year"], by_year["median_abs_pct_diff"], marker="s", color="#4C78A8", label="Median absolute close diff (%)")
ax1.set(title="Adjusted-close disagreement by year", xlabel="Calendar year", ylabel="Rows over NT$0.01 tolerance (%)")
ax2.set_ylabel("Median absolute percentage difference (%)")
ax1.legend(line1 + line2, [x.get_label() for x in line1 + line2], loc="upper right")
plt.tight_layout()
plt.show()

In [ ]:
by_stock = con.execute(f"""
SELECT
    stock_id, any_value(stock_name) AS stock_name, count(*) AS n,
    min(date) AS first_overlap_date, max(date) AS last_overlap_date,
    count(*) FILTER (WHERE abs(tej_close - finmind_close) > {ABS_PRICE_TOL}) AS close_diff_rows,
    round(100.0 * close_diff_rows / n, 3) AS close_diff_pct,
    round(median(abs(tej_close - finmind_close) / nullif(abs(finmind_close), 0)) * 100, 6) AS median_abs_pct_diff,
    round(max(abs(tej_close - finmind_close) / nullif(abs(finmind_close), 0)) * 100, 6) AS max_abs_pct_diff
FROM overlap
GROUP BY stock_id
ORDER BY median_abs_pct_diff DESC, stock_id
""").df()
display(by_stock.head(25))

In [ ]:
top_stock_plot = by_stock.head(20).sort_values("median_abs_pct_diff")
labels = top_stock_plot["stock_id"] + " " + top_stock_plot["stock_name"].fillna("")
fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(labels, top_stock_plot["median_abs_pct_diff"], color="#F58518")
ax.bar_label(bars, fmt="%.2f%%", padding=3)
ax.set(title="Stocks with the largest median adjusted-close disagreement", xlabel="Median absolute percentage difference (%)", ylabel="Stock")
plt.tight_layout()
plt.show()

## 5. 最大差異的明細

絕對差異與百分比差異會突出不同價位的股票，所以兩種排序都保留。`close_ratio_tej_to_finmind` 若在一段期間近似常數，通常表示兩邊使用了不同的公司行動調整因子或基準。

In [ ]:
largest_abs_diff = con.execute("""
SELECT stock_id, stock_name, date,
       tej_open, finmind_open, tej_high, finmind_high,
       tej_low, finmind_low, tej_close, finmind_close,
       abs(tej_close - finmind_close) AS abs_close_diff,
       round(tej_close / nullif(finmind_close, 0), 8) AS close_ratio_tej_to_finmind,
       tej_ingested_at, finmind_ingested_at, finmind_source
FROM overlap
ORDER BY abs_close_diff DESC
LIMIT 30
""").df()
display(largest_abs_diff)

In [ ]:
largest_pct_diff = con.execute("""
SELECT stock_id, stock_name, date, tej_close, finmind_close,
       round(abs(tej_close - finmind_close) / nullif(abs(finmind_close), 0) * 100, 6) AS abs_close_pct_diff,
       round(tej_close / nullif(finmind_close, 0), 8) AS close_ratio_tej_to_finmind,
       tej_ingested_at, finmind_ingested_at, finmind_source
FROM overlap
ORDER BY abs_close_pct_diff DESC NULLS LAST
LIMIT 30
""").df()
display(largest_pct_diff)

## 6. Volume、turnover、transactions

這些欄位不是 adjusted OHLC，但屬於兩邊紀錄中可合理對齊的共同欄位。TEJ 文件明確指出其 volume 與 turnover 使用千單位，以下已先換算。

In [ ]:
activity_metrics = con.execute(f"""
SELECT
    count(*) AS n,
    count(*) FILTER (WHERE tej_volume = finmind_volume) AS volume_exact,
    count(*) FILTER (WHERE abs(tej_volume - finmind_volume) <= {VOLUME_ROUND_TOL}) AS volume_within_rounding,
    round(median(abs(tej_volume - finmind_volume)), 3) AS volume_median_abs_diff,
    round(quantile_cont(abs(tej_volume - finmind_volume), 0.95), 3) AS volume_p95_abs_diff,
    max(abs(tej_volume - finmind_volume)) AS volume_max_abs_diff,
    count(*) FILTER (WHERE tej_turnover = finmind_turnover) AS turnover_exact,
    count(*) FILTER (WHERE abs(tej_turnover - finmind_turnover) <= {TURNOVER_ROUND_TOL}) AS turnover_within_rounding,
    round(median(abs(tej_turnover - finmind_turnover)), 3) AS turnover_median_abs_diff,
    round(quantile_cont(abs(tej_turnover - finmind_turnover), 0.95), 3) AS turnover_p95_abs_diff,
    max(abs(tej_turnover - finmind_turnover)) AS turnover_max_abs_diff,
    count(*) FILTER (WHERE tej_transactions = finmind_transactions) AS transactions_exact,
    count(*) FILTER (WHERE tej_transactions IS DISTINCT FROM finmind_transactions) AS transactions_different
FROM overlap
""").df()
display(activity_metrics)

## 7. 可信度診斷：公司行動造成的假性跳躍是否被消除

Backward-adjusted price 的主要用途是消除除權息、分割與減資造成的非經濟性價格跳躍。以下以 FinMind unadjusted close 找出單日原始報酬絕對值超過 20% 的事件（排除收盤為 0、間隔超過 7 天），再比較兩個 adjusted close 是否消除跳躍。這不是完整的公司行動事件研究，但比單純比較兩個來源誰的數字較接近更有判別力。

In [ ]:
con.execute("""
CREATE OR REPLACE TEMP TABLE return_diagnostics AS
WITH base AS (
    SELECT o.stock_id, o.stock_name, o.date,
           o.tej_close, o.finmind_close, r.close AS raw_close,
           lag(o.tej_close) OVER (PARTITION BY o.stock_id ORDER BY o.date) AS prev_tej_close,
           lag(o.finmind_close) OVER (PARTITION BY o.stock_id ORDER BY o.date) AS prev_finmind_close,
           lag(r.close) OVER (PARTITION BY o.stock_id ORDER BY o.date) AS prev_raw_close,
           o.date - lag(o.date) OVER (PARTITION BY o.stock_id ORDER BY o.date) AS date_gap
    FROM overlap AS o
    INNER JOIN finmind.v_daily_prices AS r USING (stock_id, date)
)
SELECT *,
       tej_close / nullif(prev_tej_close, 0) - 1 AS tej_return,
       finmind_close / nullif(prev_finmind_close, 0) - 1 AS finmind_return,
       raw_close / nullif(prev_raw_close, 0) - 1 AS raw_return
FROM base
WHERE date_gap <= 7
""")

extreme_return_summary = con.execute("""
WITH long AS (
    SELECT 'TEJ Wizard' AS dataset, tej_return AS adjusted_return FROM return_diagnostics
    UNION ALL
    SELECT 'FinMind', finmind_return FROM return_diagnostics
)
SELECT dataset, count(*) FILTER (WHERE adjusted_return IS NOT NULL) AS n_returns,
       count(*) FILTER (WHERE abs(adjusted_return) > 0.20) AS abs_gt_20pct,
       count(*) FILTER (WHERE abs(adjusted_return) > 0.50) AS abs_gt_50pct,
       count(*) FILTER (WHERE abs(adjusted_return) > 1.00) AS abs_gt_100pct,
       round(quantile_cont(abs(adjusted_return), 0.999) * 100, 3) AS p999_abs_return_pct
FROM long GROUP BY dataset ORDER BY dataset
""").df()
display(extreme_return_summary)

In [ ]:
corporate_action_proxy = con.execute("""
SELECT
    count(*) AS raw_jump_events,
    round(median(abs(raw_return)) * 100, 3) AS raw_median_abs_return_pct,
    round(median(abs(tej_return)) * 100, 3) AS tej_median_abs_return_pct,
    round(median(abs(finmind_return)) * 100, 3) AS finmind_median_abs_return_pct,
    count(*) FILTER (WHERE abs(tej_return) < abs(finmind_return)) AS tej_smoother_events,
    count(*) FILTER (WHERE abs(finmind_return) < abs(tej_return)) AS finmind_smoother_events,
    count(*) FILTER (WHERE abs(tej_return) <= 0.20) AS tej_removed_jump,
    count(*) FILTER (WHERE abs(finmind_return) <= 0.20) AS finmind_removed_jump
FROM return_diagnostics
WHERE raw_close > 0 AND prev_raw_close > 0 AND abs(raw_return) > 0.20
""").df()
display(corporate_action_proxy)

events = int(corporate_action_proxy.loc[0, "raw_jump_events"])
removed = [int(corporate_action_proxy.loc[0, "tej_removed_jump"]), int(corporate_action_proxy.loc[0, "finmind_removed_jump"])]
fig, ax = plt.subplots(figsize=(8, 4.8))
bars = ax.bar(["TEJ Wizard", "FinMind"], removed, color=["#4C78A8", "#F58518"])
ax.bar_label(bars, labels=[f"{x}/{events}" for x in removed], padding=3)
ax.set_ylim(0, events * 1.15)
ax.set(title="Large raw-price jumps reduced below 20% after adjustment", ylabel="Number of events", xlabel="Adjusted-price source")
plt.tight_layout()
plt.show()

In [ ]:
# 超過 20% 的 adjusted one-day returns 越少，越不容易殘留未處理的公司行動跳躍。
extreme_plot = extreme_return_summary.set_index("dataset")[["abs_gt_20pct", "abs_gt_50pct", "abs_gt_100pct"]]
ax = extreme_plot.plot(kind="bar", figsize=(9, 4.8), color=["#4C78A8", "#F58518", "#E45756"])
ax.set(title="Extreme one-day returns remaining in adjusted series", xlabel="Source", ylabel="Number of observations")
ax.tick_params(axis="x", rotation=0)
ax.legend(["|return| > 20%", "|return| > 50%", "|return| > 100%"], title=None)
plt.tight_layout()
plt.show()

### 個案：6669 緯穎 2026-09-02 除權

這是本次資料中很清楚的即時更新案例。FinMind unadjusted price 在除權日前約 7,000 元、除權後約 2,600 元。TEJ 已把除權前歷史價格回溯到新基準；目前 FinMind adjusted partitions 則只有 2026-08-31 起套用新因子，使序列在 2026-08-28／08-31 之間留下約 67% 的假性斷點。

In [ ]:
case_6669 = con.execute("""
SELECT o.date, o.tej_close, o.finmind_close, r.close AS finmind_raw_close
FROM overlap AS o
INNER JOIN finmind.v_daily_prices AS r USING (stock_id, date)
WHERE o.stock_id = '6669' AND o.date BETWEEN DATE '2026-08-20' AND DATE '2026-09-03'
ORDER BY o.date
""").df()
display(case_6669)

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(case_6669["date"], case_6669["tej_close"], marker="o", label="TEJ adjusted close", color="#4C78A8")
ax.plot(case_6669["date"], case_6669["finmind_close"], marker="s", label="FinMind adjusted close", color="#F58518")
ax.plot(case_6669["date"], case_6669["finmind_raw_close"], marker="^", linestyle="--", label="FinMind raw close", color="#79706E")
ax.axvline(pd.Timestamp("2026-09-02"), color="#E45756", linestyle=":", label="Ex-right date")
ax.set(title="6669 Wiwynn: adjustment-basis transition around the 2026 ex-right event", xlabel="Date", ylabel="Close (NTD)")
ax.legend()
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 8. 最近共同日期的 sanity check

最新日期通常較少受歷史調整因子差異影響。這裡確認最新共同日期的 OHLC 是否一致。

In [ ]:
latest_date_check = con.execute(f"""
WITH latest AS (SELECT max(date) AS d FROM overlap)
SELECT date, count(*) AS n,
       count(*) FILTER (WHERE tej_open = finmind_open AND tej_high = finmind_high
                         AND tej_low = finmind_low AND tej_close = finmind_close) AS exact_rows,
       count(*) FILTER (WHERE abs(tej_open - finmind_open) <= {ABS_PRICE_TOL}
                         AND abs(tej_high - finmind_high) <= {ABS_PRICE_TOL}
                         AND abs(tej_low - finmind_low) <= {ABS_PRICE_TOL}
                         AND abs(tej_close - finmind_close) <= {ABS_PRICE_TOL}) AS within_tolerance_rows
FROM overlap, latest
WHERE date = latest.d
GROUP BY date
""").df()
display(latest_date_check)

## 9. 選擇性輸出差異明細

預設不寫檔。把 `EXPORT_RESULTS` 改成 `True` 後，會輸出 OHLC 超過容差的完整 Parquet，以及數個小型 CSV summary。

In [ ]:
EXPORT_RESULTS = False
EXPORT_DIR = ROOT / "data-pipelines/tej-wizard/notebooks/outputs/tej_finmind_adj_price_comparison"

if EXPORT_RESULTS:
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    coverage.to_csv(EXPORT_DIR / "coverage.csv", index=False, encoding="utf-8-sig")
    ohlc_metrics.to_csv(EXPORT_DIR / "ohlc_metrics.csv", index=False, encoding="utf-8-sig")
    by_year.to_csv(EXPORT_DIR / "close_diff_by_year.csv", index=False, encoding="utf-8-sig")
    by_stock.to_csv(EXPORT_DIR / "close_diff_by_stock.csv", index=False, encoding="utf-8-sig")

    detail_path = (EXPORT_DIR / "ohlc_differences_over_tolerance.parquet").as_posix()
    con.execute(f"""
    COPY (
        SELECT *,
               tej_open - finmind_open AS open_diff,
               tej_high - finmind_high AS high_diff,
               tej_low - finmind_low AS low_diff,
               tej_close - finmind_close AS close_diff
        FROM overlap
        WHERE abs(tej_open - finmind_open) > {ABS_PRICE_TOL}
           OR abs(tej_high - finmind_high) > {ABS_PRICE_TOL}
           OR abs(tej_low - finmind_low) > {ABS_PRICE_TOL}
           OR abs(tej_close - finmind_close) > {ABS_PRICE_TOL}
        ORDER BY stock_id, date
    ) TO '{detail_path}' (FORMAT PARQUET, COMPRESSION ZSTD)
    """)
    print(f"已輸出到：{EXPORT_DIR}")
else:
    print("EXPORT_RESULTS=False；本次未寫出檔案。")

## 10. 分析報告與可信度結論（資料快照：2026-09-03，TEJ 側 120 檔 universe）

用本 notebook 的查詢邏輯驗證目前 repo 內資料後：

- TEJ 有 651,118 筆 / 120 檔；FinMind 有 11,226,843 筆 / 3,152 檔。
- 兩邊有 119 檔共同股票、639,157 筆相同股票且相同日期的紀錄，範圍為 2000-09-04 至 2026-09-03；兩邊 join key 都沒有重複。
- 只有 16,074 筆（2.5149%）四個 OHLC 完全一致；若容許每個價位相差不超過 0.01 元，384,195 筆（60.1096%）四價皆一致，仍有 254,962 筆超過容差。
- close 的絕對差中位數是 0.006099 元，相對差中位數 0.019531%；但第 95 百分位相對差為 24.580572%，顯示分布有明顯長尾，不只是小數位精度。
- 最新共同日期 2026-09-03 有 107 筆，四個 OHLC 全部完全一致。歷史差異遠高於近期，符合公司行動調整方法或調整基準不同的型態。
- 最大絕對 close 差出現在 6669 緯穎 2026-08-28：TEJ 2,413.836、FinMind 7,200，兩者比例約 0.335255；到 2026-08-31 後又接近一致，呈現明顯的調整因子切換。
- 最大相對 close 差樣本集中在 2357 華碩的 2009–2010 年，約 346.61%。這類極端值應回到公司行動定義與來源更新時點查核，不能只視為浮點誤差。
- TEJ 有但 FinMind adjusted price 完全沒有的唯一一檔是 3012 廣輝（2002-07-30 至 2006-09-19）。
- 單位換算後，transactions 有 638,484 / 639,157 筆完全一致；volume 與 turnover 的中位絕對差分別是 482 股與 500 元。

### 哪個來源較可信？

**就目前這兩個本地 DB、共同股票與日期，以及「用於總報酬／回測的 backward-adjusted price」這個目的，TEJ Wizard 較可信。** 這是有範圍限制的判斷，不代表 TEJ 的每一筆資料都必然正確，也不代表 FinMind 的原始成交價較差。

主要證據：

1. 在 55 個 FinMind unadjusted close 單日跳動超過 20% 的事件中，TEJ adjusted close 將 40 個事件壓回 20% 內，FinMind 只有 18 個；兩者的事件日 adjusted return 絕對值中位數分別是 4.908% 與 20.968%。TEJ 在 34 個事件較平滑，FinMind 在 20 個較平滑。
2. 在相同日期序列中，TEJ adjusted close 殘留 15 個超過 20% 的單日跳動，FinMind 有 48 個；超過 50% 分別為 2 與 10 個，超過 100% 分別為 1 與 4 個。台股 adjusted series 留下越多這類極端斷點，越可能仍混有未回溯的公司行動調整。
3. 6669 緯穎是直接可見的近期案例：FinMind raw close 從除權前約 7,000 元切換到除權後約 2,600 元；TEJ 已將除權前歷史值一併回溯，FinMind adjusted 則只從較新的日期套用新因子，造成序列中仍有約 67% 的假性跳空。
4. TEJ 公開說明明確定義其資料以最新股價為基準向回調整，並納入配息、配股、增減資與分割；FinMind 公開資料集說明確認 `TaiwanStockPriceAdj` 是除權息調整後、供長期報酬使用，但公開頁面提供的公式與基準細節較少。可稽核性上 TEJ 較完整。
5. FinMind adjusted 資料在此 warehouse 有明顯存活者偏誤：已知 767 檔下市證券只有 7 檔有 adjusted history。TEJ 本身只涵蓋曾進入 0050 歷史成分的 120 檔，不能當全市場資料，但在這個限定 universe 裡保留退場證券較完整；本次唯一完全無法交集的是 3012 廣輝。

### 使用建議

- 長期報酬、技術指標或歷史回測：在 TEJ 這 120 檔涵蓋範圍內，以 TEJ adjusted OHLC 為主。
- 當日真實成交價、成交量與全市場廣度：使用 unadjusted daily-price 表，不要把 adjusted close 當實際成交價；若需要 120 檔以外的股票，FinMind 覆蓋較廣，但公司行動日前後必須額外檢查。
- 若仍使用 FinMind adjusted：每次除權息／分割後重抓該股票的完整歷史，且不要讓舊的逐日 market snapshot 覆蓋已更新的 ticker history；並加入「單日 adjusted return 超過 20%」的自動告警。

### 方法與外部文件

- [TEJ：還原股價原則與計算方式](https://www.tejwin.com/insight/%E9%82%84%E5%8E%9F%E8%82%A1%E5%83%B9/)
- [TEJ：調整股價與報酬率公式](https://www.tejwin.com/qa-category/%E4%BA%A4%E6%98%93%E9%9D%A2/)
- [FinMind 官方資料集說明：TaiwanStockPriceAdj](https://github.com/FinMind/FinMind-MCP/blob/master/knowledge/datasets.md)

> 結論適用於本 repo 在 2026-09-03 的資料快照。adjusted history 會隨新公司行動回寫，資料更新後應重新執行全部 cell。

In [ ]:
con.close()